<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-02-embeddings-and-vectors/lesson-2.3-firestore-vector/notebooks/GCP_Capstone_2.3_Firestore_Vector.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2.3 Firestore Vector Search — Your First Vector Database
**Netsetos GenAI Engineering — GCP Capstone**

Store embeddings with Vector(), query with find_nearest(), build a complete RAG pipeline.


## Setup


In [ ]:
!pip install -q google-cloud-firestore==2.30.0 google-genai==2.21.0
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS
FIRESTORE_LOCATION = 'asia-south1'  # Mumbai - the course rule for DocuMind's stored data
CHUNKS_COLLECTION = 'chunks'        # THE collection: 2.3 seeds it, 4.x reads it, 12.5 writes it in production
TENANT_ID = 'acme'                  # every document carries a tenant; every query filters on it
EMBED_MODEL = 'text-embedding-005'  # ONE embedding model per store (deploy/services/rag-api/config.py)

import json, os, re, subprocess
from google.cloud.firestore_v1.base_query import FieldFilter   # every tenant filter in this lesson (first used in Cell 4)

# Enable the APIs this lesson calls (idempotent, quiet). Lesson 1.1 already enabled them for
# the capstone project; on a fresh project the databases create below fails without this.
subprocess.run(['gcloud', 'services', 'enable', 'firestore.googleapis.com',
                'aiplatform.googleapis.com', '--project', PROJECT_ID, '--quiet'], check=False)

# Firestore needs a (default) database in Native mode; enabling the API is not enough.
# COURSE RULE: DocuMind's (default) database is created in asia-south1 (Mumbai), and that
# location is PERMANENT - it cannot be changed after creation. Embeddings still come from the
# us-central1 client and generation from the global endpoint; data residency covers data at
# rest, not generation traffic.
if '(default)' not in subprocess.run(
        ['gcloud', 'firestore', 'databases', 'list', '--project', PROJECT_ID, '--format=value(name)'],
        capture_output=True, text=True).stdout:
    print(f'Creating Firestore (default) database in {FIRESTORE_LOCATION} (one-time, ~30s)...')
    subprocess.run(['gcloud', 'firestore', 'databases', 'create',
                    '--location=' + FIRESTORE_LOCATION, '--project', PROJECT_ID], check=False)
else:
    print('Firestore (default) database ready.')

from google.cloud import firestore
from google.cloud.firestore_v1.vector import Vector
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure
from google import genai
from google.genai import types

db = firestore.Client(project=PROJECT_ID)
ai = genai.Client(enterprise=True, project=PROJECT_ID, location='us-central1')  # embeddings: regional-only
gen_client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')  # generation: global endpoint (Gemini 3.x)
collection = db.collection(CHUNKS_COLLECTION)
print('Clients initialized')


## Cell 1: Create Vector Indexes

Firestore vector search needs composite indexes. **Run the next cell** to create both (works in Colab, not only Cloud Shell):
- `tenant_id` + `embedding` — the tenant-filtered `find_nearest()` (Cell 4): there is no unfiltered query on a multi-tenant store
- `tenant_id` + `doc_type` + `embedding` — **filtered** search by document type (Cells 5-7): the handbook (`policy`) or the law it must comply with (`statute`)

Each index takes **~2-5 minutes to build**. If a `find_nearest()` cell raises `FAILED_PRECONDITION: Missing vector index`, the index is still building (or a filtered query needs its own composite index) — wait a bit and re-run.

In [ ]:
# Firestore vector search needs composite VECTOR INDEXES. This lesson creates two:
#   1. tenant_id (ASC) + embedding            -> THE production index, byte-for-byte what
#      deploy/terraform/firestore_indexes.tf provisions. Equality filter FIRST, vector LAST.
#   2. tenant_id (ASC) + doc_type (ASC) + embedding -> this lesson's filtered demo (Cells 5-7)
# There is deliberately NO unfiltered index: on a multi-tenant store an unfiltered query is a
# cross-tenant read, and the absence of the index is what makes it impossible.
# gcloud works in Colab; idempotent (ignores "already exists"). Each index takes
# ~2-5 min to BUILD before the matching query works.
import subprocess
_BASE = (f"gcloud firestore indexes composite create --project={PROJECT_ID} "
         "--collection-group=" + CHUNKS_COLLECTION + " --query-scope=COLLECTION ")
_VEC = "--field-config=field-path=embedding,vector-config='{\"dimension\":\"768\",\"flat\":\"{}\"}'"
_TENANT = "--field-config=field-path=tenant_id,order=ascending "
_INDEXES = {
    'tenant_id + embedding (production)': _TENANT + _VEC,
    'tenant_id + doc_type + embedding (filtered)': _TENANT + "--field-config=field-path=doc_type,order=ascending " + _VEC,
}
for _name, _fields in _INDEXES.items():
    _r = subprocess.run(_BASE + _fields, shell=True, capture_output=True, text=True)
    _out = (_r.stdout + _r.stderr).strip()
    if _r.returncode == 0:
        print(f'[{_name}] creation started.')
    elif 'already exists' in _out.lower():
        print(f'[{_name}] already exists.')
    else:
        print(f'[{_name}] {_out}')


### Wait for the vector index to build (run after Setup)
This blocks until the Firestore vector index reports `READY` — about 2-5 minutes on the **first** run only (returns instantly afterwards). The retrieval cells below raise `FAILED_PRECONDITION` until this prints READY. Safe to re-run any time.

In [ ]:
# Wait until the indexes finish building - find_nearest() raises FAILED_PRECONDITION
# ("Missing vector index") until every index is READY, not just created.
import subprocess, time
print('Waiting for indexes to build (~2-5 min; find_nearest fails until READY)...')
for _ in range(40):  # up to ~10 min
    _states = subprocess.run(
        f"gcloud firestore indexes composite list --project={PROJECT_ID} --format='value(state)'",
        shell=True, capture_output=True, text=True).stdout.split()
    if _states and 'CREATING' not in _states:
        print(f'All {len(_states)} indexes READY - find_nearest cells (4-7) will work now.'); break
    time.sleep(15)
else:
    print('Still building after ~10 min. Check: '
          f'!gcloud firestore indexes composite list --project={PROJECT_ID}')

## Cell 2: Store a Single Document with Embedding

One real chunk first. The document is DocuMind's own corpus — `deploy/evals/corpus/` in the course repo: ACME's synthetic employee handbook, MSA, invoice and annual report, and **thirteen real documents** (the four Labour Codes, the Payment of Bonus, Gratuity, Maternity Benefit, POSH, DPDP, IT and CGST Acts, and the ministry's compliance handbook) fetched from the publishers' own sites by `deploy/evals/fetch_real.py`, sha256 of every file in `real_sources.json`. The loader below is `deploy/shared/documind_corpus.py`, pasted verbatim (`tools/check_contract.py` holds this cell to it): the ingest worker's chunker and the eval gate's chunk ids, so what you store here is what DocuMind serves in Module 12, not a lookalike.


In [ ]:
# DocuMind's corpus, through the kit's loader - deploy/shared/documind_corpus.py pasted VERBATIM
# (found beside this notebook, or cloned under /content the way 10.4 does). One chunker, one id
# scheme (acme:hr_policy_2026#NP-03), from this cell to the ingest worker and the eval gate.
# --- kit: begin ---------------------------------------------------------------
KIT_REPO = "https://github.com/netsetos/agentic-ai-weekend-gcp-learners"   # the learner repo carries the kit under deploy/
KIT_BRANCH = "main"   # the learner repo (public): the notebooks and the kit, deploy/, on its main branch
CHUNK_CHARS, CHUNK_OVERLAP = 2000, 200                     # services/ingest/main.py
_SECTION = re.compile(r"^## +(.+?) *$", re.M)
_CODE = re.compile(r"^([A-Z][A-Z0-9]{0,7}(?:-[A-Z0-9]{1,6}){1,2})\b")   # NP-03, IT-SEC-04, MSA-04, GEN-014


def find_kit(start: str = ".") -> str:
    """The deploy/evals directory: beside the notebook, above it, or a clone under /content."""
    here = os.path.abspath(start)
    for _ in range(6):
        for cand in (os.path.join(here, "deploy", "evals"), os.path.join(here, "evals"), here):
            if os.path.isfile(os.path.join(cand, "manifest.json")) and os.path.isdir(os.path.join(cand, "corpus")):
                return cand
        here = os.path.dirname(here)
    clone = "/content/agentic-ai-weekend-gcp-learners"
    if not os.path.isdir(clone):
        subprocess.run(["git", "clone", "--depth", "1", "-b", KIT_BRANCH, KIT_REPO, clone], check=True)
    return os.path.join(clone, "deploy", "evals")


def load_documents(tenant: str, evals_dir: str, project_id: str) -> list:
    """Every document of one tenant that has text on disk: the synthetic .md files and the real
    Acts' pypdf mirrors. A scanned PDF with no mirror (posh_act_2013) is skipped - that one is
    lesson 4.1's, and only Document AI can read it."""
    docs = []
    for m in json.load(open(os.path.join(evals_dir, "manifest.json"), encoding="utf-8")):
        if m["tenant_id"] != tenant or not m.get("chars"):
            continue
        mirror = os.path.join(evals_dir, m["file"].rsplit(".", 1)[0] + ".md")
        if not os.path.isfile(mirror):
            continue
        docs.append({"slug": m["slug"], "doc_type": m["doc_type"],
                     "source_uri": m["gcs_uri"].replace("${PROJECT_ID}", project_id),
                     "text": open(mirror, encoding="utf-8").read()})
    return docs


def windows(text: str) -> list:
    """The worker's chunker: fixed windows with overlap, ending on a sentence when one is nearby."""
    text = text.strip()
    out, start = [], 0
    while start < len(text):
        end = min(len(text), start + CHUNK_CHARS)
        if end < len(text):
            cut = text.rfind(". ", start + CHUNK_CHARS // 2, end)
            if cut != -1:
                end = cut + 1
        piece = text[start:end].strip()
        if piece:
            out.append(piece)
        if end >= len(text):
            break
        start = max(end - CHUNK_OVERLAP, start + 1)
    return out


def chunk_document(doc: dict, tenant: str) -> list:
    """Canonical chunk documents - the fields services/ingest/indexer.py writes - minus the embedding."""
    text = re.sub(r"\A\s*<!--.*?-->\s*", "", doc["text"], count=1, flags=re.S)   # a mirror's provenance header
    base = {"tenant_id": tenant, "source_uri": doc["source_uri"], "doc_type": doc["doc_type"], "kind": "text"}
    out = []
    heads = list(_SECTION.finditer(text))
    if heads:                                                  # a handbook: one chunk per section
        for n, h in enumerate(heads):
            body = text[h.end(): heads[n + 1].start() if n + 1 < len(heads) else len(text)].strip()
            title = h.group(1).strip()
            code = _CODE.match(title)
            key = code.group(1) if code else f"s{n}"
            for k, piece in enumerate(windows(f"{title}\n{body}")):
                cid = f"{tenant}:{doc['slug']}#{key}" + (f"-{k}" if k else "")
                out.append({**base, "chunk_id": cid, "text": piece, "page_start": 1, "section": title})
    else:                                                      # a PDF mirror: pages split by \f
        for p, page in enumerate(text.split("\f"), 1):
            for k, piece in enumerate(windows(page)):
                out.append({**base, "chunk_id": f"{tenant}:{doc['slug']}#p{p}-{k}",
                            "text": piece, "page_start": p})
    return out


EMBED_BATCH, EMBED_TOKENS, CHARS_PER_TOKEN = 250, 15_000, 3


def embed_batches(texts: list) -> list:
    """Batches of at most EMBED_BATCH texts AND about EMBED_TOKENS tokens. text-embedding-005 takes
    250 texts per request and 20,000 tokens across them, and a request over either limit fails
    whole; a two-thousand-character chunk is ~500 tokens, so 250 of them are ~125,000. The first
    live corpus load (6 Sept 2026) failed every long Act exactly here - the same rule now lives in
    services/ingest/indexer.py."""
    out, cur, cur_tokens = [], [], 0
    for t in texts:
        tokens = max(1, len(t) // CHARS_PER_TOKEN)
        if cur and (len(cur) >= EMBED_BATCH or cur_tokens + tokens > EMBED_TOKENS):
            out.append(cur); cur, cur_tokens = [], 0
        cur.append(t); cur_tokens += tokens
    if cur:
        out.append(cur)
    return out


def seed(db, embed, tenant: str, project_id: str, evals_dir: str = None, collection: str = "chunks") -> dict:
    """Write one tenant's corpus into Firestore, idempotently. A document that already has a chunk
    under this tenant - written by this loader or by 4.1's Document AI path - is skipped, so two
    lessons never hold two copies of one document. `embed(texts) -> vectors` is the notebook's
    batched text-embedding-005 call. Returns {slug: chunks written}."""
    from google.cloud import firestore
    from google.cloud.firestore_v1.base_query import FieldFilter
    from google.cloud.firestore_v1.vector import Vector
    evals_dir = evals_dir or find_kit()
    counts = {}
    for doc in load_documents(tenant, evals_dir, project_id):
        chunks = chunk_document(doc, tenant)
        present = (db.collection(collection).where(filter=FieldFilter("tenant_id", "==", tenant))
                   .where(filter=FieldFilter("source_uri", "==", doc["source_uri"])).limit(1).get())
        if not chunks or present:
            counts[doc["slug"]] = 0
            continue
        vectors = []
        for batch in embed_batches([c["text"] for c in chunks]):   # 250 texts AND 20,000 tokens per request
            vectors += embed(batch)
        batch, n = db.batch(), 0
        for c, v in zip(chunks, vectors):
            batch.set(db.collection(collection).document(c["chunk_id"]),
                      {**c, "embedding": Vector(v), "processed_at": firestore.SERVER_TIMESTAMP})
            n += 1
            if n % 400 == 0:                                     # a Firestore batch holds 500 writes
                batch.commit()
                batch = db.batch()
        batch.commit()
        counts[doc["slug"]] = len(chunks)
    return counts


## Cell 3: Batch Ingest DocuMind's Corpus

The rest of the corpus, in batches: text-embedding-005 takes up to 250 texts per request (2.2's batch rule) and a Firestore batch commits at most 500 writes. `seed()` does both and skips a document the tenant already holds, so it is safe to re-run. About 1,600 chunks for ACME and $0.05 of embeddings on a first run.


In [ ]:
# Cell 2 stored ONE chunk of the handbook, and seed() skips a document the tenant already holds -
# so delete that chunk first and the whole handbook loads (on a re-run everything is skipped).
collection.document(clause['chunk_id']).delete()

def embed_documents(texts):
    """Batched: up to 250 texts per request (2.2's batch rule). Cell 2 embedded one at a time."""
    return [e.values for e in ai.models.embed_content(
        model=EMBED_MODEL, contents=texts,
        config=types.EmbedContentConfig(task_type='RETRIEVAL_DOCUMENT', output_dimensionality=768)
    ).embeddings]

written = seed(db, embed_documents, TENANT_ID, PROJECT_ID, EVALS_DIR, CHUNKS_COLLECTION)
for _slug, _n in written.items():
    print(f'  {_slug:38} {_n:4d} chunks' + ('' if _n else '   (already present)'))
print(f'Loaded {sum(written.values())} chunks for tenant {TENANT_ID!r} '
      f'(the scanned POSH Act has no text layer; lesson 4.1 reads it with Document AI)')


## Cell 4: find_nearest() — Vector Search


In [ ]:
query_text = 'What is the notice period for a confirmed employee?'
q_resp = ai.models.embed_content(
    model=EMBED_MODEL, contents=query_text,
    config=types.EmbedContentConfig(task_type='RETRIEVAL_QUERY', output_dimensionality=768))

# Tenant filter FIRST. There is no unfiltered query on this store - the index does not exist.
results = collection.where(filter=FieldFilter('tenant_id', '==', TENANT_ID)).find_nearest(
    vector_field='embedding',
    query_vector=Vector(q_resp.embeddings[0].values),
    distance_measure=DistanceMeasure.COSINE,
    limit=5,
    distance_result_field='vector_distance',
).get()

print(f'Query: {query_text}')
for doc in results:
    d = doc.to_dict()
    sim = 1 - d['vector_distance']
    print(f'  [{sim:.4f}] {d.get("doc_type",""):8} {doc.id:40} {d["text"][:70]!r}')


## Cell 5: Filtered Vector Search


In [ ]:
from google.cloud.firestore_v1.base_query import FieldFilter

# Search ONLY this tenant's statutes - the real Acts and Codes - for a question the handbook cannot
# answer. NOTE: requires the composite index (tenant_id + doc_type + embedding) from Cell 1.
q_statute = ai.models.embed_content(
    model=EMBED_MODEL, contents='After how many years of continuous service is gratuity payable?',
    config=types.EmbedContentConfig(task_type='RETRIEVAL_QUERY', output_dimensionality=768)).embeddings[0].values
results = (collection.where(filter=FieldFilter('tenant_id', '==', TENANT_ID))
                     .where(filter=FieldFilter('doc_type', '==', 'statute'))).find_nearest(
    vector_field='embedding',
    query_vector=Vector(q_statute),
    distance_measure=DistanceMeasure.COSINE,
    limit=3,
    distance_result_field='vector_distance',
).get()

print('Filtered search (statutes only):')
for doc in results:
    d = doc.to_dict()
    print(f'  [{1-d["vector_distance"]:.4f}] {doc.id} p.{d.get("page_start")}  {d["text"][:80]!r}')


## Cell 6: Complete RAG Pipeline


In [ ]:
def rag_query(question, doc_type=None, top_k=3):
    # Embed query
    q_emb = ai.models.embed_content(
        model=EMBED_MODEL, contents=question,
        config=types.EmbedContentConfig(task_type='RETRIEVAL_QUERY', output_dimensionality=768)
    ).embeddings[0].values
    
    # Search
    ref = collection.where(filter=FieldFilter('tenant_id', '==', TENANT_ID))   # always
    if doc_type:
        ref = ref.where(filter=FieldFilter('doc_type', '==', doc_type))
    docs = ref.find_nearest(
        vector_field='embedding', query_vector=Vector(q_emb),
        distance_measure=DistanceMeasure.COSINE,
        limit=top_k, distance_result_field='dist',
        distance_threshold=0.5,
    ).get()
    
    context = '\n'.join([d.to_dict()['text'] for d in docs])
    
    # Generate
    response = gen_client.models.generate_content(
        model='gemini-3.6-flash',
        contents=f'Answer ONLY from the context below. If it does not contain the answer, say so.\n\nContext:\n{context}\n\nQuestion: {question}',
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level="LOW")))
    
    print(f'Query: {question}')
    print(f'Sources: {len(docs)} documents')
    for d in docs:
        dd = d.to_dict()
        print(f'  [{1-dd["dist"]:.4f}] {dd["text"][:60]}...')
    print(f'\nAnswer: {response.text}')

rag_query('What is the notice period for a confirmed employee?')
print('\n' + '='*60 + '\n')
rag_query('After how many years of continuous service is gratuity payable?', doc_type='statute')   # section 4(1) of the Payment of Gratuity Act, 1972


## Cell 7: FirestoreRAG Production Module


In [ ]:
from google.cloud import firestore
from google.cloud.firestore_v1.vector import Vector
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure
from google.cloud.firestore_v1.base_query import FieldFilter
from google import genai
from google.genai import types

EMBED_MODEL = "text-embedding-005"   # ONE model per store
GEN_MODEL = "gemini-3.6-flash"
DIMS = 768

class FirestoreRAG:
    def __init__(self, project, collection_name=CHUNKS_COLLECTION, tenant_id=TENANT_ID):
        self.db = firestore.Client(project=project)
        self.ai = genai.Client(enterprise=True, project=project, location="us-central1")  # embeddings: regional-only
        self.gen = genai.Client(enterprise=True, project=project, location="global")  # generation: global endpoint
        self.collection = self.db.collection(collection_name)
        self.tenant_id = tenant_id

    def embed(self, text, task="RETRIEVAL_DOCUMENT"):
        r = self.ai.models.embed_content(
            model=EMBED_MODEL, contents=text,
            config=types.EmbedContentConfig(task_type=task, output_dimensionality=DIMS))
        return r.embeddings[0].values

    def ingest(self, documents):
        """Ingest list of dicts with 'text' and optional 'doc_type'. Writes the canonical
        document - the shape services/ingest/indexer.py writes in production."""
        batch = self.db.batch()
        pending = 0
        for i, doc in enumerate(documents):
            vec = self.embed(doc["text"], task="RETRIEVAL_DOCUMENT")
            ref = self.collection.document(f"{self.tenant_id}:doc_{i:04d}")   # the tenant in the id: two tenants' doc_0001 are two documents (4.8, F23)
            batch.set(ref, {"tenant_id": self.tenant_id, "text": doc["text"],
                            "source_uri": doc.get("source_uri", f"gs://{PROJECT_ID}-uploads/{self.tenant_id}/doc_{i:04d}.md"),
                            "page_start": doc.get("page_start"), "doc_type": doc.get("doc_type", "policy"),
                            "kind": "text",   # the seventh field indexer.py writes
                            "embedding": Vector(vec)})
            pending += 1
            if pending == 500:  # Firestore commits at most 500 writes per batch
                batch.commit()
                batch = self.db.batch()
                pending = 0
            print(f"  embedded {i+1}/{len(documents)}", end="\r")
        if pending:
            batch.commit()
        return len(documents)

    def search(self, query, category=None, top_k=5, threshold=0.5):
        q_vec = self.embed(query, task="RETRIEVAL_QUERY")
        ref = self.collection.where(filter=FieldFilter('tenant_id', '==', self.tenant_id))
        if category:
            ref = ref.where(filter=FieldFilter("doc_type", "==", category))
        docs = ref.find_nearest(
            vector_field="embedding", query_vector=Vector(q_vec),
            distance_measure=DistanceMeasure.COSINE,
            limit=top_k, distance_result_field="dist",
            distance_threshold=threshold).get()
        return [{"chunk_id": d.id, "text": d.to_dict()["text"],
                 "source_uri": d.to_dict().get("source_uri", ""),
                 "page_start": d.to_dict().get("page_start"),
                 "doc_type": d.to_dict().get("doc_type", ""),
                 "similarity": 1 - d.to_dict()["dist"]} for d in docs]

    def query(self, question, category=None):
        results = self.search(question, category)
        context = "\n".join(
            f"[Source {n}] {r['text']}" for n, r in enumerate(results, 1))
        response = self.gen.models.generate_content(
            model=GEN_MODEL,
            contents=f"Context:\n{context}\n\nQuestion: {question}",
            config=types.GenerateContentConfig(thinking_config=types.ThinkingConfig(thinking_level="LOW")))
        return {"answer": response.text, "sources": results}

# Test: query() end-to-end against the corpus loaded in Cell 3 - a real Act, a real page
rag = FirestoreRAG(PROJECT_ID)
out = rag.query('What is the overtime rate under the Code on Wages?')
print(out['answer'])
for s in out['sources']:
    print(f"  [{s['similarity']:.4f}] {s['chunk_id']} p.{s['page_start']}  {s['text'][:80]!r}")


## ✅ Lesson 2.3 Complete!

- ✅ Created flat vector index via gcloud
- ✅ Stored DocuMind's corpus — the tenant's documents and thirteen real documents — as canonical documents with Vector() embeddings, through the one loader every seeding lesson shares
- ✅ Queried with find_nearest() + COSINE distance
- ✅ Converted cosine distance → similarity
- ✅ Filtered vector search with .where()
- ✅ distance_threshold for quality filtering
- ✅ Full RAG pipeline: embed → search → generate
- ✅ FirestoreRAG production module

**Next: Lesson 2.4 — AlloyDB pgvector & BigQuery Vector Search**
